# Polarization of Electromagnetic Waves

A monochromatic plane wave travelling along $z$ has two independent transverse degrees of freedom:

$$\mathbf{E}(z,t)=\hat{\mathbf{x}}\,A_x\cos(\omega t-kz)+\hat{\mathbf{y}}\,A_y\cos(\omega t-kz+\delta)$$

The polarization state is fixed entirely by the amplitude ratio $A_y/A_x$ and the relative phase $\delta$ — the frequency, the wavelength and the absolute amplitude play no role. Every figure in this notebook is generated from those two numbers.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
    "axes.titlesize": 10,
})

CX, CY, CE, CM = "#1f77b4", "#d62728", "#111111", "#7f4fbf"   # Ex, Ey, E, auxiliary
SL = {"style": {"description_width": "78px"},
      "layout": widgets.Layout(width="330px"), "continuous_update": False}


def field(Ax, Ay, delta, phase):
    """Transverse components at a given phase wt - kz."""
    return Ax * np.cos(phase), Ay * np.cos(phase + delta)


def stokes(Ax, Ay, delta):
    """S0..S3 for a fully polarized state."""
    return np.array([Ax**2 + Ay**2,
                     Ax**2 - Ay**2,
                     2 * Ax * Ay * np.cos(delta),
                     2 * Ax * Ay * np.sin(delta)])


def ellipse_params(Ax, Ay, delta):
    """Orientation psi, ellipticity chi (rad), semi-axes a, b."""
    S0, S1, S2, S3 = stokes(Ax, Ay, delta)
    S0 = max(S0, 1e-12)
    psi = 0.5 * np.arctan2(S2, S1)
    chi = 0.5 * np.arcsin(np.clip(S3 / S0, -1.0, 1.0))
    A = np.sqrt(S0)
    return psi, chi, A * np.cos(chi), A * abs(np.sin(chi))


def hand_label(delta):
    s = np.sin(delta)
    if abs(s) < 1e-6:
        return "linear"
    return "RH optics / LH IEEE" if s > 0 else "LH optics / RH IEEE"


def style_transverse(ax, lim):
    """Common formatting for every transverse-plane panel."""
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.axhline(0, color="0.6", lw=0.6); ax.axvline(0, color="0.6", lw=0.6)
    ax.set_xlabel("$E_x$"); ax.set_ylabel("$E_y$")


print("helpers ready — E_x, E_y, Stokes, ellipse geometry")

helpers ready — E_x, E_y, Stokes, ellipse geometry


## 1 — The polarization ellipse

Eliminating time from the two components leaves the locus traced by the tip of $\mathbf{E}$ in the transverse plane:

$$\left(\frac{E_x}{A_x}\right)^{2}+\left(\frac{E_y}{A_y}\right)^{2}-2\,\frac{E_xE_y}{A_xA_y}\cos\delta=\sin^{2}\delta$$

Viewing convention used everywhere below: we look **back along the propagation axis at the oncoming wave**, so $+z$ points out of the screen, $x$ is right and $y$ is up.

| $\delta$ | $A_y/A_x$ | resulting state |
|---|---|---|
| $0$ or $\pm180^\circ$ | any | linear, tilted by $\arctan(A_y/A_x)$ |
| $\pm 90^\circ$ | $1$ | circular |
| $\pm 90^\circ$ | $\neq 1$ | ellipse aligned with the $x,y$ axes |
| anything else | any | tilted ellipse |

Sweep `delta` with `wt` held fixed to morph the shape; then hold `delta` and sweep `wt` to watch the field vector run around it.

In [2]:
def draw_ellipse(Ax, Ay, delta_deg, wt_deg):
    delta, wt = np.deg2rad(delta_deg), np.deg2rad(wt_deg)
    lim = 1.15
    ph = np.linspace(0, 2 * np.pi, 400)
    ex, ey = field(Ax, Ay, delta, ph)
    cx, cy = field(Ax, Ay, delta, wt)
    nx, ny = field(Ax, Ay, delta, wt + 0.25)          # a step later in time

    fig = plt.figure(figsize=(12.6, 3.9))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1, 1.5], wspace=0.32)

    # time traces of the two components
    a0 = fig.add_subplot(gs[0])
    pd = np.linspace(-180, 540, 720)
    tx, ty = field(Ax, Ay, delta, np.deg2rad(pd))
    a0.plot(pd, tx, color=CX, lw=1.6, label="$E_x$")
    a0.plot(pd, ty, color=CY, lw=1.6, label="$E_y$")
    a0.axvline(wt_deg, color=CE, lw=1.2, ls="--")
    a0.plot([wt_deg], [cx], "o", color=CX, ms=6)
    a0.plot([wt_deg], [cy], "o", color=CY, ms=6)
    if Ay > 0.02 and Ax > 0.02:                        # mark the phase offset
        a0.annotate("", xy=(0, 1.06), xytext=(-delta_deg, 1.06),
                    arrowprops=dict(arrowstyle="<->", color=CM, lw=1.2))
        a0.text(-delta_deg / 2, 1.12, f"$\\delta$ = {delta_deg:.0f}°",
                color=CM, ha="center", fontsize=9)
    a0.set_xlim(-180, 540); a0.set_ylim(-1.25, 1.3)
    a0.set_xticks(np.arange(-180, 541, 180))
    a0.set_xlabel("phase  $\\omega t - kz$  [deg]"); a0.set_ylabel("amplitude")
    a0.legend(fontsize=8, loc="lower right", ncol=2)
    a0.set_title("components")

    # transverse plane
    a1 = fig.add_subplot(gs[1])
    style_transverse(a1, lim)
    a1.plot(ex, ey, color="0.55", lw=1.2)
    a1.plot([cx, cx], [0, cy], color=CY, lw=0.8, ls=":")
    a1.plot([0, cx], [cy, cy], color=CX, lw=0.8, ls=":")
    a1.plot([cx], [0], "o", color=CX, ms=5)
    a1.plot([0], [cy], "o", color=CY, ms=5)
    a1.annotate("", xy=(cx, cy), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=CE, lw=2))
    a1.annotate("", xy=(nx, ny), xytext=(cx, cy),
                arrowprops=dict(arrowstyle="-|>", color=CM, lw=2.2))
    a1.set_title(f"tip of $\\mathbf{{E}}$  —  {hand_label(delta)}")

    # snapshot in space
    a2 = fig.add_subplot(gs[2], projection="3d")
    z = np.linspace(0, 2, 400)
    px, py = field(Ax, Ay, delta, wt - 2 * np.pi * z)
    a2.plot(z, px, py, color=CE, lw=1.6)
    zs = z[::20]
    for zz, xx, yy in zip(zs, px[::20], py[::20]):
        a2.plot([zz, zz], [0, xx], [0, yy], color="0.75", lw=0.7)
    a2.plot(z, px, -lim, color=CX, lw=1.0, alpha=0.8)      # floor: Ex(z)
    a2.plot(z, np.full_like(z, lim), py, color=CY, lw=1.0, alpha=0.8)
    a2.plot(np.zeros_like(ph), ex, ey, color="0.55", lw=1.0, ls="--")
    a2.set_xlim(0, 2); a2.set_ylim(-lim, lim); a2.set_zlim(-lim, lim)
    a2.set_xlabel("$z/\\lambda$"); a2.set_ylabel("$E_x$"); a2.set_zlabel("$E_y$")
    a2.set_box_aspect([2.4, 1, 1]); a2.view_init(elev=18, azim=-62)
    a2.set_title("snapshot along the propagation axis")
    plt.show()


w = dict(Ax=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                description="Aₓ:", **SL),
         Ay=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                description="A_y:", **SL),
         delta_deg=widgets.FloatSlider(value=90, min=-180, max=180, step=5,
                                       description="δ [deg]:", **SL),
         wt_deg=widgets.FloatSlider(value=0, min=0, max=360, step=5,
                                    description="ωt [deg]:", **SL))
ui = widgets.VBox([widgets.HBox([w["Ax"], w["Ay"]]),
                   widgets.HBox([w["delta_deg"], w["wt_deg"]])])
display(ui, widgets.interactive_output(draw_ellipse, w))

Output()

## 2 — Orientation and ellipticity

Two angles replace the pair $(A_y/A_x,\ \delta)$ and describe the ellipse geometrically: the orientation $\psi$ of the major axis and the ellipticity angle $\chi$.

$$\tan 2\psi=\frac{2A_xA_y\cos\delta}{A_x^{2}-A_y^{2}},\qquad
\sin 2\chi=\frac{2A_xA_y\sin\delta}{A_x^{2}+A_y^{2}}$$

With $A=\sqrt{A_x^{2}+A_y^{2}}$ the semi-axes are $a=A\cos\chi$ and $b=A\,|\sin\chi|$, so $\tan|\chi|=b/a$. The sign of $\chi$ carries the handedness: $\chi=0$ is linear, $\chi=\pm45^\circ$ is circular.

Note how $\psi$ jumps by $90^\circ$ when $A_x=A_y$ and $\delta$ crosses $\pm90^\circ$ — the major and minor axes swap roles there, which is why $\psi$ alone is a poor descriptor near circular states.

In [3]:
def draw_geometry(Ax, Ay, delta_deg):
    delta = np.deg2rad(delta_deg)
    psi, chi, a, b = ellipse_params(Ax, Ay, delta)
    lim = 1.5
    ph = np.linspace(0, 2 * np.pi, 400)
    ex, ey = field(Ax, Ay, delta, ph)

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(11.4, 4.1),
                                 gridspec_kw={"width_ratios": [1, 1.35]})

    a0.plot(ex, ey, color=CE, lw=1.8)
    a0.add_patch(plt.Rectangle((-Ax, -Ay), 2 * Ax, 2 * Ay, fill=False,
                               ec="0.7", ls=":", lw=1))                 # bounding box
    u = np.array([np.cos(psi), np.sin(psi)])
    v = np.array([-np.sin(psi), np.cos(psi)])
    a0.plot([-a * u[0], a * u[0]], [-a * u[1], a * u[1]], color=CM, lw=1.6)
    a0.plot([-b * v[0], b * v[0]], [-b * v[1], b * v[1]], color=CX, lw=1.6)
    arc = np.linspace(0, psi, 60)
    a0.plot(0.42 * np.cos(arc), 0.42 * np.sin(arc), color=CM, lw=1.2)
    a0.text(0.5 * np.cos(psi / 2), 0.5 * np.sin(psi / 2), "$\\psi$",
            color=CM, fontsize=11)
    a0.text(a * u[0] * 0.55, a * u[1] * 0.55 + 0.1, "$a$", color=CM, fontsize=10)
    a0.text(b * v[0] * 0.6, b * v[1] * 0.6, "$b$", color=CX, fontsize=10)
    style_transverse(a0, lim)
    a0.set_title(f"ψ = {np.rad2deg(psi):+.1f}°   χ = {np.rad2deg(chi):+.1f}°   "
                 f"b/a = {b / max(a, 1e-9):.2f}\n{hand_label(delta)}")

    d = np.linspace(-180, 180, 721)
    pp, cc = [], []
    for dv in np.deg2rad(d):
        p_, c_, _, _ = ellipse_params(Ax, Ay, dv)
        pp.append(np.rad2deg(p_)); cc.append(np.rad2deg(c_))
    a1.plot(d, pp, color=CM, lw=1.6, label="$\\psi$  orientation")
    a1.plot(d, cc, color=CY, lw=1.6, label="$\\chi$  ellipticity")
    a1.axvline(delta_deg, color=CE, lw=1.2, ls="--")
    a1.plot([delta_deg], [np.rad2deg(psi)], "o", color=CM, ms=6)
    a1.plot([delta_deg], [np.rad2deg(chi)], "o", color=CY, ms=6)
    a1.axhline(0, color="0.6", lw=0.6)
    a1.set_xlim(-180, 180); a1.set_xticks(np.arange(-180, 181, 90))
    a1.set_xlabel("δ [deg]"); a1.set_ylabel("angle [deg]")
    a1.legend(fontsize=8); a1.set_title("both angles vs relative phase")
    plt.tight_layout(); plt.show()


w2 = dict(Ax=widgets.FloatSlider(value=1.0, min=0.05, max=1.0, step=0.05,
                                 description="Aₓ:", **SL),
          Ay=widgets.FloatSlider(value=0.55, min=0.05, max=1.0, step=0.05,
                                 description="A_y:", **SL),
          delta_deg=widgets.FloatSlider(value=50, min=-180, max=180, step=5,
                                        description="δ [deg]:", **SL))
display(widgets.HBox([w2["Ax"], w2["Ay"], w2["delta_deg"]]),
        widgets.interactive_output(draw_geometry, w2))

Output()

## 3 — Jones calculus: acting on the state

A fully polarized state is a complex 2-vector, with the physical field recovered as $\mathbf{E}(t)=\mathrm{Re}\{\mathbf{J}e^{i\omega t}\}$:

$$\mathbf{J}=\begin{pmatrix}A_x\\ A_ye^{i\delta}\end{pmatrix},\qquad
\mathbf{J}_{\text{out}}=M\,\mathbf{J}_{\text{in}}$$

Optical elements are $2\times2$ matrices, built by rotating a diagonal form into the element's frame with $R(\theta)$:

$$P(\theta)=R(\theta)\begin{pmatrix}1&0\\0&0\end{pmatrix}R(-\theta),\qquad
W(\theta,\Gamma)=R(\theta)\begin{pmatrix}1&0\\0&e^{-i\Gamma}\end{pmatrix}R(-\theta)$$

A polarizer projects and throws away power — for linear input this is Malus' law, $I/I_0=\cos^{2}(\theta-\psi)$. A retarder is unitary: it conserves power and only reshapes the ellipse. $\Gamma=90^\circ$ gives a quarter-wave plate (linear $\leftrightarrow$ circular), $\Gamma=180^\circ$ a half-wave plate, which mirrors the state about the fast axis and so rotates a linear input by $2(\theta-\psi)$.

In [ ]:
def rot(t):
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])


def element(kind, theta):
    R, Ri = rot(theta), rot(-theta)
    if kind == "linear polarizer":
        core = np.diag([1.0, 0.0]).astype(complex)
    elif kind == "quarter-wave plate":
        core = np.diag([1.0, np.exp(-1j * np.pi / 2)])
    elif kind == "half-wave plate":
        core = np.diag([1.0, np.exp(-1j * np.pi)])
    else:
        core = np.eye(2, dtype=complex)
    return R @ core @ Ri


def unpack(J):
    d = np.angle(J[1]) - np.angle(J[0])
    return abs(J[0]), abs(J[1]), (d + np.pi) % (2 * np.pi) - np.pi


def plot_state(ax, Ax, Ay, delta, lim, color, title):
    ph = np.linspace(0, 2 * np.pi, 300)
    ex, ey = field(Ax, Ay, delta, ph)
    ax.plot(ex, ey, color=color, lw=1.8)
    cx, cy = field(Ax, Ay, delta, 0.0)
    nx, ny = field(Ax, Ay, delta, 0.25)
    ax.annotate("", xy=(nx, ny), xytext=(cx, cy),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2))
    style_transverse(ax, lim)
    ax.set_title(title)


def draw_jones(Ax, Ay, delta_deg, kind, theta_deg):
    delta, theta = np.deg2rad(delta_deg), np.deg2rad(theta_deg)
    Jin = np.array([Ax + 0j, Ay * np.exp(1j * delta)])
    Jout = element(kind, theta) @ Jin
    Ax2, Ay2, d2 = unpack(Jout)
    I0 = float(np.vdot(Jin, Jin).real)
    I1 = float(np.vdot(Jout, Jout).real)
    lim = 1.5

    fig, (a0, a1, a2) = plt.subplots(1, 3, figsize=(12.4, 4.0))
    plot_state(a0, Ax, Ay, delta, lim, CX, f"input   I = {I0:.3f}")
    if kind != "none":                                   # draw the element axis
        for ax_ in (a0, a1):
            ax_.plot([-lim * np.cos(theta), lim * np.cos(theta)],
                     [-lim * np.sin(theta), lim * np.sin(theta)],
                     color=CM, lw=1.2, ls="--")
        a0.text(lim * 0.62 * np.cos(theta), lim * 0.62 * np.sin(theta) + 0.08,
                "axis", color=CM, fontsize=8)
    if I1 > 1e-9:
        plot_state(a1, Ax2, Ay2, d2, lim, CY,
                   f"output   I = {I1:.3f}   ({I1 / max(I0, 1e-12):.0%})")
    else:
        style_transverse(a1, lim)
        a1.plot([0], [0], "x", color=CY, ms=12, mew=2)
        a1.set_title("output   I = 0   (fully extinguished)")

    th = np.linspace(0, 180, 361)
    if kind == "linear polarizer" or kind == "none":
        y = [float(np.vdot(element("linear polarizer", np.deg2rad(t)) @ Jin,
                           element("linear polarizer", np.deg2rad(t)) @ Jin).real)
             for t in th]
        a2.plot(th, np.array(y) / max(I0, 1e-12), color=CM, lw=1.6)
        a2.set_ylabel("$I/I_0$"); a2.set_ylim(-0.05, 1.05)
        a2.set_title("Malus' law" if kind != "none"
                     else "Malus' law (if a polarizer were inserted)")
    else:
        y = []
        for t in th:
            _, c_, _, _ = ellipse_params(*unpack(element(kind, np.deg2rad(t)) @ Jin))
            y.append(np.rad2deg(c_))
        a2.plot(th, y, color=CM, lw=1.6)
        a2.axhline(0, color="0.6", lw=0.6)
        a2.set_ylabel("output χ [deg]"); a2.set_ylim(-50, 50)
        a2.set_title("output ellipticity vs plate angle")
    if kind != "none":
        a2.axvline(theta_deg % 180, color=CE, lw=1.2, ls="--")
    a2.set_xlim(0, 180); a2.set_xticks(np.arange(0, 181, 45))
    a2.set_xlabel("element angle θ [deg]")
    plt.tight_layout(); plt.show()


w3 = dict(Ax=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                 description="Aₓ:", **SL),
          Ay=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                 description="A_y:", **SL),
          delta_deg=widgets.FloatSlider(value=0, min=-180, max=180, step=5,
                                        description="δ [deg]:", **SL),
          kind=widgets.Dropdown(options=["none", "linear polarizer",
                                         "quarter-wave plate", "half-wave plate"],
                                value="quarter-wave plate", description="element:",
                                style={"description_width": "78px"},
                                layout=widgets.Layout(width="330px")),
          theta_deg=widgets.FloatSlider(value=0, min=0, max=180, step=1,
                                        description="θ [deg]:", **SL))
display(widgets.VBox([widgets.HBox([w3["Ax"], w3["Ay"], w3["delta_deg"]]),
                      widgets.HBox([w3["kind"], w3["theta_deg"]])]),
        widgets.interactive_output(draw_jones, w3))

Output()

## 4 — Stokes parameters and the Poincaré sphere

Jones vectors cannot describe partially polarized light. The Stokes parameters can, because they are built from **time-averaged intensities**, all four measurable with a detector and a few elements:

$$S_0=\langle E_x^2\rangle+\langle E_y^2\rangle,\quad
S_1=\langle E_x^2\rangle-\langle E_y^2\rangle,\quad
S_2=2\langle E_xE_y\rangle_{0},\quad
S_3=2\langle E_xE_y\rangle_{90}$$

For a fully polarized wave $S_1^2+S_2^2+S_3^2=S_0^2$, so the normalized triple lands on the unit **Poincaré sphere** at longitude $2\psi$ and latitude $2\chi$: equator = linear, poles = circular. The degree of polarization

$$\text{DoP}=\frac{\sqrt{S_1^2+S_2^2+S_3^2}}{S_0}\in[0,1]$$

is the radius, so partially polarized light sits *inside* the ball and unpolarized light sits at the centre. Drag `DoP` below 1 and watch the state fall off the surface — that is exactly the information a single Jones vector cannot carry.

In [ ]:
def draw_poincare(Ax, Ay, delta_deg, dop):
    delta = np.deg2rad(delta_deg)
    S = stokes(Ax, Ay, delta)
    S0 = max(S[0], 1e-12)
    s = dop * S[1:] / S0                       # normalized, depolarized
    psi, chi, _, _ = ellipse_params(Ax, Ay, delta)

    fig = plt.figure(figsize=(11.6, 4.4))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    u = np.linspace(0, 2 * np.pi, 25)
    v = np.linspace(0, np.pi, 13)
    ax.plot_wireframe(np.outer(np.cos(u), np.sin(v)),
                      np.outer(np.sin(u), np.sin(v)),
                      np.outer(np.ones_like(u), np.cos(v)),
                      color="0.85", lw=0.35)
    t = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), 0, color="0.4", lw=1.4)          # linear equator
    ax.plot(np.cos(t), 0, np.sin(t), color="0.75", lw=0.9)         # meridians
    ax.plot(0, np.cos(t), np.sin(t), color="0.75", lw=0.9)
    for e in ([1, 0, 0], [0, 1, 0], [0, 0, 1]):
        e = np.array(e)
        ax.plot(*np.array([-e, e]).T, color="0.6", lw=0.7, ls=":")
    for vec, lab in [((1.42, 0, 0), "H"), ((-1.42, 0, 0), "V"),
                     ((0, 1.42, 0), "+45°"), ((0, -1.42, 0), "−45°"),
                     ((0, 0, 1.3), "RH"), ((0, 0, -1.3), "LH")]:
        ax.text(*vec, lab, fontsize=8, color=CM, ha="center")
    ax.plot([s[0], s[0]], [s[1], s[1]], [0, s[2]],
            color=CY, lw=0.9, ls="--")                             # latitude drop
    ax.plot([0, s[0]], [0, s[1]], [0, s[2]], color=CE, lw=1.8)
    ax.scatter([s[0]], [s[1]], [s[2]], color=CY, s=70,
               edgecolor="k", linewidth=0.6, depthshade=False)
    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15); ax.set_zlim(-1.15, 1.15)
    ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1]); ax.set_zticks([-1, 0, 1])
    ax.tick_params(labelsize=7, pad=0)
    ax.set_xlabel("$S_1/S_0$", labelpad=-4); ax.set_ylabel("$S_2/S_0$", labelpad=-4)
    ax.set_zlabel("$S_3/S_0$", labelpad=-4)
    ax.set_box_aspect([1, 1, 1]); ax.view_init(elev=20, azim=38)
    ax.grid(False)
    ax.set_title(f"Poincaré sphere   DoP = {dop:.2f}")

    a1 = fig.add_subplot(1, 2, 2)
    a1.bar(["$S_1/S_0$", "$S_2/S_0$", "$S_3/S_0$"], s,
           color=[CX, CM, CY], width=0.55)
    a1.axhline(0, color="0.4", lw=0.8)
    a1.set_ylim(-1.05, 1.05); a1.set_ylabel("normalized Stokes")
    a1.set_title(f"$S_0$ = {S0:.3f}   2ψ = {np.rad2deg(2 * psi):+.0f}°   "
                 f"2χ = {np.rad2deg(2 * chi):+.0f}°")
    for i, val in enumerate(s):
        a1.text(i, val + (0.06 if val >= 0 else -0.1), f"{val:+.2f}",
                ha="center", fontsize=9)
    plt.tight_layout(); plt.show()


w4 = dict(Ax=widgets.FloatSlider(value=1.0, min=0.05, max=1.0, step=0.05,
                                 description="Aₓ:", **SL),
          Ay=widgets.FloatSlider(value=0.7, min=0.05, max=1.0, step=0.05,
                                 description="A_y:", **SL),
          delta_deg=widgets.FloatSlider(value=60, min=-180, max=180, step=5,
                                        description="δ [deg]:", **SL),
          dop=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05,
                                  description="DoP:", **SL))
display(widgets.VBox([widgets.HBox([w4["Ax"], w4["Ay"]]),
                      widgets.HBox([w4["delta_deg"], w4["dop"]])]),
        widgets.interactive_output(draw_poincare, w4))

Output()

## 5 — The circular basis

Linear and circular are not different physical categories, only different bases. Any state decomposes into two counter-rotating circular components:

$$\mathbf{E}(t)=a_R\begin{pmatrix}\cos(\omega t-\varphi/2)\\ -\sin(\omega t-\varphi/2)\end{pmatrix}
+a_L\begin{pmatrix}\cos(\omega t+\varphi/2)\\ \sin(\omega t+\varphi/2)\end{pmatrix}$$

For $a_R=a_L$ the two vertical components cancel at every instant and the sum is **linear**, oriented at $\psi=\varphi/2$. So a phase difference between the circular components rotates the plane of polarization by half as much — the mechanism behind optical activity and Faraday rotation. When $a_R\neq a_L$ the cancellation is incomplete and the resultant opens into an ellipse with $\tan\chi=(a_L-a_R)/(a_L+a_R)$; setting one amplitude to zero returns a pure circular state.

In [6]:
def draw_circular(aR, aL, phi_deg, wt_deg):
    phi, wt = np.deg2rad(phi_deg), np.deg2rad(wt_deg)
    t = np.linspace(0, 2 * np.pi, 400)

    def comps(p):
        r = np.array([aR * np.cos(p - phi / 2), -aR * np.sin(p - phi / 2)])
        l = np.array([aL * np.cos(p + phi / 2), aL * np.sin(p + phi / 2)])
        return r, l

    R_t, L_t = comps(t)
    R_n, L_n = comps(wt)
    tot = R_t + L_t
    lim = 1.15 * max(aR + aL, 0.1)

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(10.4, 4.3))

    a0.plot(*R_t, color=CX, lw=1.1, ls=":", label="$a_R$ clockwise")
    a0.plot(*L_t, color=CY, lw=1.1, ls=(0, (5, 3)), label="$a_L$ counter-cw")
    a0.legend(fontsize=7, loc="lower left")
    a0.annotate("", xy=tuple(R_n), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=CX, lw=1.8))
    a0.annotate("", xy=tuple(R_n + L_n), xytext=tuple(R_n),
                arrowprops=dict(arrowstyle="-|>", color=CY, lw=1.8))
    a0.annotate("", xy=tuple(R_n + L_n), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=CE, lw=2.4))
    a0.plot(*tot, color="0.6", lw=1.2)
    style_transverse(a0, lim)
    a0.set_title("clockwise (blue) + counter-clockwise (red) = resultant")

    a1.plot(*tot, color=CE, lw=1.8)
    ang = np.linspace(0, np.pi, 200)
    if abs(aR - aL) < 1e-9 and aR > 0:                 # exactly linear
        psi = phi / 2
        a1.plot([-lim * np.cos(psi), lim * np.cos(psi)],
                [-lim * np.sin(psi), lim * np.sin(psi)], color=CM, lw=1.2, ls="--")
        a1.plot(0.35 * np.cos(ang[ang <= max(psi, 1e-6)]),
                0.35 * np.sin(ang[ang <= max(psi, 1e-6)]), color=CM, lw=1.2)
        note = f"linear,  ψ = φ/2 = {np.rad2deg(psi):.0f}°"
    else:
        chi = np.arctan2(aL - aR, aL + aR)
        note = f"elliptical,  χ = {np.rad2deg(chi):+.1f}°   b/a = {abs(np.tan(chi)):.2f}"
    style_transverse(a1, lim)
    a1.set_title(note)
    plt.tight_layout(); plt.show()


w5 = dict(aR=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                 description="a_R:", **SL),
          aL=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                 description="a_L:", **SL),
          phi_deg=widgets.FloatSlider(value=0, min=-180, max=180, step=5,
                                      description="φ [deg]:", **SL),
          wt_deg=widgets.FloatSlider(value=0, min=0, max=360, step=5,
                                     description="ωt [deg]:", **SL))
display(widgets.VBox([widgets.HBox([w5["aR"], w5["aL"]]),
                      widgets.HBox([w5["phi_deg"], w5["wt_deg"]])]),
        widgets.interactive_output(draw_circular, w5))

Output()